## Khai báo thư viện và thiết lập môi trường

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN KAGGLE
# ==============================================================================
DATA_DIR = '/kaggle/input/datasets/tunoquc/dataaa'
MODEL_DIR = '/kaggle/working/outputs'
os.makedirs(MODEL_DIR, exist_ok=True)

# ==============================================================================
# 2. CẤU HÌNH SIÊU THAM SỐ & TÌM KIẾM 
# ==============================================================================
OPTUNA_N_TRIALS = 50

# Không gian tìm kiếm Optuna (Task 3 & 6)
SPACE_LR_MIN, SPACE_LR_MAX = 0.01, 0.3
SPACE_ESTIMATORS_MIN, SPACE_ESTIMATORS_MAX = 100, 1000

XGB_DEPTH_MIN, XGB_DEPTH_MAX = 3, 10
XGB_SUBSAMPLE_MIN, XGB_SUBSAMPLE_MAX = 0.6, 1.0
XGB_COLSAMPLE_MIN, XGB_COLSAMPLE_MAX = 0.6, 1.0

LGB_LEAVES_MIN, LGB_LEAVES_MAX = 20, 150
LGB_MIN_CHILD_MIN, LGB_MIN_CHILD_MAX = 10, 100

# ==============================================================================
# 3. THÔNG SỐ CHỐT TỪ BÁO CÁO 
# ==============================================================================
# Báo cáo 6.5: XGBoost Final Params
XGB_FINAL_PARAMS = {
    'n_estimators': 750,
    'learning_rate': 0.045,
    'max_depth': 7,
    'subsample': 0.82,
    'colsample_bytree': 0.75,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist'
}

# Báo cáo 6.6: LightGBM Final Params
LGB_FINAL_PARAMS = {
    'n_estimators': 820,
    'learning_rate': 0.038,
    'num_leaves': 95,
    'min_child_samples': 22,
    'objective': 'binary',
    'metric': 'auc',
    'is_unbalance': True,
    'verbose': -1
}

# Báo cáo 6.7: CatBoost Final Params
CAT_ITERATIONS = 500
CAT_LEARNING_RATE = 0.05
CAT_DEPTH = 6

# ==============================================================================
# 4. HÀM TỐI ƯU NGƯỠNG 
# ==============================================================================
def optimize_threshold(y_true, y_probs):
    """
    Quét qua các ngưỡng từ 0.1 đến 0.9 để tìm ngưỡng cho F1-Score cao nhất.
    """
    thresholds = np.arange(0.1, 0.91, 0.01)
    best_thresh = 0.5
    best_f1 = 0
    
    for t in thresholds:
        preds = (y_probs >= t).astype(int)
        f1 = f1_score(y_true, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
            
    return best_thresh, best_f1

print("--- Đã tải cấu hình và hàm tiện ích thành công ---")

--- Đã tải cấu hình và hàm tiện ích thành công ---


## Load dữ liệu đã tiền xử lý & Meta Info

In [2]:
# Load meta_info từ bước 02_preprocessing_fe
with open(f"{DATA_DIR}/meta_info.pkl", "rb") as f:
    meta_info = pickle.load(f)

SCALE_POS_WEIGHT = meta_info.get('scale_pos_weight', 1.0)
N_FOLDS = meta_info.get('n_folds', 5)
RANDOM_STATE = meta_info.get('random_seed', 42)

print(f"Meta Info Loaded: scale_pos_weight={SCALE_POS_WEIGHT:.4f}, n_folds={N_FOLDS}, random_state={RANDOM_STATE}")

# Load bộ dữ liệu dành riêng cho Tree-based models
X = pd.read_pickle(f"{DATA_DIR}/train_tree_ready.pkl")
y = pd.read_pickle(f"{DATA_DIR}/y_train.pkl")

# Hold-out 20% cho Validation Validation như mô tả trong section 6.2
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Val: {X_val.shape}")

Meta Info Loaded: scale_pos_weight=4.5032, n_folds=5, random_state=42
Kích thước tập Train: (112560, 33)
Kích thước tập Val: (28140, 33)


## Task 3 - Tối ưu siêu tham số XGBoost với Optuna

In [3]:
def objective_xgb(trial):
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'tree_method': 'hist',
        'random_state': RANDOM_STATE,
        'scale_pos_weight': SCALE_POS_WEIGHT,
        'learning_rate': trial.suggest_float('learning_rate', SPACE_LR_MIN, SPACE_LR_MAX, log=True),
        'max_depth': trial.suggest_int('max_depth', XGB_DEPTH_MIN, XGB_DEPTH_MAX),
        'subsample': trial.suggest_float('subsample', XGB_SUBSAMPLE_MIN, XGB_SUBSAMPLE_MAX),
        'colsample_bytree': trial.suggest_float('colsample_bytree', XGB_COLSAMPLE_MIN, XGB_COLSAMPLE_MAX),
        'n_estimators': trial.suggest_int('n_estimators', SPACE_ESTIMATORS_MIN, SPACE_ESTIMATORS_MAX)
    }

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        model = xgb.XGBClassifier(**param)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        
        preds = model.predict_proba(X_va)[:, 1]
        score = roc_auc_score(y_va, preds)
        cv_scores.append(score)

    return np.mean(cv_scores)

print("--- Đang khởi chạy Optuna Tuning cho XGBoost...")
study_xgb = optuna.create_study(direction='maximize', study_name="XGBoost_Tuning")
study_xgb.optimize(objective_xgb, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)
print(f"Best CV AUC: {study_xgb.best_value:.5f}")

[I 2026-04-16 14:03:18,856] A new study created in memory with name: XGBoost_Tuning


--- Đang khởi chạy Optuna Tuning cho XGBoost...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-16 14:03:37,288] Trial 0 finished with value: 0.9753129563857608 and parameters: {'learning_rate': 0.050370172731848725, 'max_depth': 3, 'subsample': 0.6005112692404105, 'colsample_bytree': 0.9672590863665547, 'n_estimators': 471}. Best is trial 0 with value: 0.9753129563857608.
[I 2026-04-16 14:04:12,196] Trial 1 finished with value: 0.9703787361394051 and parameters: {'learning_rate': 0.22485924953887326, 'max_depth': 4, 'subsample': 0.6444284807820596, 'colsample_bytree': 0.9921903619982222, 'n_estimators': 843}. Best is trial 0 with value: 0.9753129563857608.
[I 2026-04-16 14:05:04,653] Trial 2 finished with value: 0.967333568343283 and parameters: {'learning_rate': 0.2961309022514341, 'max_depth': 7, 'subsample': 0.8259884116029399, 'colsample_bytree': 0.7796484279816636, 'n_estimators': 884}. Best is trial 0 with value: 0.9753129563857608.
[I 2026-04-16 14:06:17,933] Trial 3 finished with value: 0.9709568548706166 and parameters: {'learning_rate': 0.07807535261486023, 

## Task 4 - Huấn luyện XGBoost

In [4]:
print("--- Đang huấn luyện XGBoost Final Model...")

# Lấy params từ cấu hình báo cáo và cập nhật thêm cấu hình hệ thống
xgb_final_cfg = XGB_FINAL_PARAMS.copy()
xgb_final_cfg['random_state'] = RANDOM_STATE
xgb_final_cfg['scale_pos_weight'] = SCALE_POS_WEIGHT

xgb_final = xgb.XGBClassifier(**xgb_final_cfg)
xgb_final.fit(X_train, y_train)

# Đánh giá & Tối ưu Threshold
y_val_pred_prob_xgb = xgb_final.predict_proba(X_val)[:, 1]
xgb_val_auc = roc_auc_score(y_val, y_val_pred_prob_xgb)

best_t_xgb, best_f1_xgb = optimize_threshold(y_val, y_val_pred_prob_xgb)

print(f"--- Kết quả Đánh giá XGBoost ---")
print(f"Validation AUC-ROC : {xgb_val_auc:.5f}")
print(f"Optimal Threshold  : {best_t_xgb:.2f}")
print(f"Validation F1-Score: {best_f1_xgb:.5f}")

with open(os.path.join(MODEL_DIR, "xgb_final.pkl"), "wb") as f:
    pickle.dump(xgb_final, f)

--- Đang huấn luyện XGBoost Final Model...
--- Kết quả Đánh giá XGBoost ---
Validation AUC-ROC : 0.97209
Optimal Threshold  : 0.66
Validation F1-Score: 0.82722


## Task 6 - Tối ưu siêu tham số LightGBM với Optuna

In [5]:
def objective_lgb(trial):
    param = {
        'objective': 'binary',
        'metric': 'auc',
        'random_state': RANDOM_STATE,
        'is_unbalance': True, 
        'verbose': -1,        
        'learning_rate': trial.suggest_float('learning_rate', SPACE_LR_MIN, SPACE_LR_MAX, log=True),
        'num_leaves': trial.suggest_int('num_leaves', LGB_LEAVES_MIN, LGB_LEAVES_MAX),
        'min_child_samples': trial.suggest_int('min_child_samples', LGB_MIN_CHILD_MIN, LGB_MIN_CHILD_MAX),
        'n_estimators': trial.suggest_int('n_estimators', SPACE_ESTIMATORS_MIN, SPACE_ESTIMATORS_MAX)
    }

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[valid_idx], y_train.iloc[valid_idx]

        model = lgb.LGBMClassifier(**param)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
        
        preds = model.predict_proba(X_va)[:, 1]
        score = roc_auc_score(y_va, preds)
        cv_scores.append(score)

    return np.mean(cv_scores)

print("--- Đang khởi chạy Optuna Tuning cho LightGBM...")
study_lgb = optuna.create_study(direction='maximize', study_name="LightGBM_Tuning")
study_lgb.optimize(objective_lgb, n_trials=OPTUNA_N_TRIALS, show_progress_bar=True)
print(f"Best CV AUC: {study_lgb.best_value:.5f}")

[I 2026-04-16 14:23:19,459] A new study created in memory with name: LightGBM_Tuning


--- Đang khởi chạy Optuna Tuning cho LightGBM...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-16 14:23:28,919] Trial 0 finished with value: 0.9700306250534323 and parameters: {'learning_rate': 0.27469446760966876, 'num_leaves': 124, 'min_child_samples': 57, 'n_estimators': 109}. Best is trial 0 with value: 0.9700306250534323.
[I 2026-04-16 14:23:56,252] Trial 1 finished with value: 0.9712797753956576 and parameters: {'learning_rate': 0.14688771332548778, 'num_leaves': 150, 'min_child_samples': 97, 'n_estimators': 317}. Best is trial 1 with value: 0.9712797753956576.
[I 2026-04-16 14:24:41,776] Trial 2 finished with value: 0.9726930953810549 and parameters: {'learning_rate': 0.044755245554382525, 'num_leaves': 106, 'min_child_samples': 32, 'n_estimators': 674}. Best is trial 2 with value: 0.9726930953810549.
[I 2026-04-16 14:25:34,736] Trial 3 finished with value: 0.969770332617314 and parameters: {'learning_rate': 0.24697315157205207, 'num_leaves': 145, 'min_child_samples': 83, 'n_estimators': 666}. Best is trial 2 with value: 0.9726930953810549.
[I 2026-04-16 14:26:

## Task 7 - Huấn luyện LightGBM

In [6]:
print("--- Đang huấn luyện LightGBM Final Model...")

lgb_final_cfg = LGB_FINAL_PARAMS.copy()
lgb_final_cfg['random_state'] = RANDOM_STATE

lgb_final = lgb.LGBMClassifier(**lgb_final_cfg)
lgb_final.fit(X_train, y_train)

# Đánh giá & Tối ưu Threshold
y_val_pred_prob_lgb = lgb_final.predict_proba(X_val)[:, 1]
lgb_val_auc = roc_auc_score(y_val, y_val_pred_prob_lgb)

best_t_lgb, best_f1_lgb = optimize_threshold(y_val, y_val_pred_prob_lgb)

print(f"--- Kết quả Đánh giá LightGBM ---")
print(f"Validation AUC-ROC : {lgb_val_auc:.5f}")
print(f"Optimal Threshold  : {best_t_lgb:.2f}")
print(f"Validation F1-Score: {best_f1_lgb:.5f}")

with open(os.path.join(MODEL_DIR, "lgb_final.pkl"), "wb") as f:
    pickle.dump(lgb_final, f)

--- Đang huấn luyện LightGBM Final Model...
--- Kết quả Đánh giá LightGBM ---
Validation AUC-ROC : 0.97255
Optimal Threshold  : 0.69
Validation F1-Score: 0.82533


## Task 8 - Huấn luyện CatBoost

In [7]:
print("--- Đang huấn luyện CatBoost Final Model...")

cat_final = CatBoostClassifier(
    iterations=CAT_ITERATIONS,
    learning_rate=CAT_LEARNING_RATE,
    depth=CAT_DEPTH,
    auto_class_weights='Balanced',
    eval_metric='AUC',
    random_seed=RANDOM_STATE,
    verbose=100 
)

cat_final.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

# Đánh giá & Tối ưu Threshold
y_val_pred_prob_cat = cat_final.predict_proba(X_val)[:, 1]
cat_val_auc = roc_auc_score(y_val, y_val_pred_prob_cat)

best_t_cat, best_f1_cat = optimize_threshold(y_val, y_val_pred_prob_cat)

print(f"--- Kết quả Đánh giá CatBoost ---")
print(f"Validation AUC-ROC : {cat_val_auc:.5f}")
print(f"Optimal Threshold  : {best_t_cat:.2f}")
print(f"Validation F1-Score: {best_f1_cat:.5f}")

with open(os.path.join(MODEL_DIR, "cat_final.pkl"), "wb") as f:
    pickle.dump(cat_final, f)

--- Đang huấn luyện CatBoost Final Model...
0:	test: 0.9534806	best: 0.9534806 (0)	total: 76.6ms	remaining: 38.2s
100:	test: 0.9742075	best: 0.9742075 (100)	total: 2.07s	remaining: 8.17s
200:	test: 0.9745847	best: 0.9745889 (193)	total: 4.01s	remaining: 5.97s
300:	test: 0.9746817	best: 0.9746966 (283)	total: 6.01s	remaining: 3.97s
400:	test: 0.9746317	best: 0.9746966 (283)	total: 7.96s	remaining: 1.97s
499:	test: 0.9746249	best: 0.9746966 (283)	total: 9.91s	remaining: 0us

bestTest = 0.9746966484
bestIteration = 283

Shrink model to first 284 iterations.
--- Kết quả Đánh giá CatBoost ---
Validation AUC-ROC : 0.97470
Optimal Threshold  : 0.73
Validation F1-Score: 0.83208
